In [ ]:
import pandas as pd
import os
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
import re
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from itertools import dropwhile
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
import time
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.layers import LSTM


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'twitter-entity-sentiment-analysis' dataset.
Path to dataset files: /kaggle/input/twitter-entity-sentiment-analysis


In [ ]:
import os

print(os.listdir(path))

['twitter_validation.csv', 'twitter_training.csv']


In [ ]:
train_path = os.path.join(path, "twitter_training.csv")
valid_path = os.path.join(path, "twitter_validation.csv")

train_df = pd.read_csv(
    train_path,
    header=None,
    names=["Tweet_ID", "Entity", "Sentiment", "Tweet_content"]
)

valid_df = pd.read_csv(
    valid_path,
    header=None,
    names=["Tweet_ID", "Entity", "Sentiment", "Tweet_content"]
)

In [ ]:
#drop unnecessary columns
drop_columns = ["Tweet_ID", "Entity"]
train_df = train_df.drop(drop_columns, axis=1)
valid_df = valid_df.drop(drop_columns, axis=1)

In [ ]:
print(train_df.head())

  Sentiment                                      Tweet_content
0  Positive  im getting on borderlands and i will murder yo...
1  Positive  I am coming to the borders and I will kill you...
2  Positive  im getting on borderlands and i will kill you ...
3  Positive  im coming on borderlands and i will murder you...
4  Positive  im getting on borderlands 2 and i will murder ...


In [ ]:
print(len(train_df))

74682


In [ ]:
print(train_df["Sentiment"].value_counts())

Sentiment
Negative      22542
Positive      20832
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [ ]:
def preprocess_text_data(text):


  #normalize
  text = text.lower()

  #remove digits, punctuation, non alphabetic characters because they don't contribute to the checking, tags, URLs
  text = re.sub(r"http\S+|@\w+|#\w+|\d+|[^A-Za-z\s]", " ", text)

  #word tokenization because i think emails spam checking doesn't need context it rather checks words
  tokens = word_tokenize(text)

  #remove stop words
  tokens = [word for word in tokens if word not in stop_words]

  #lemmatization
  tokens = [lemmatizer.lemmatize(word) for word in tokens]



  clean_text = " ".join(tokens)


  return clean_text


In [ ]:
print(train_df["Tweet_content"].isna().sum())
print(valid_df["Tweet_content"].isna().sum())

686
0


In [ ]:
train_df = train_df.dropna(subset=["Tweet_content"])

In [ ]:
train_df["clean_text"] = train_df["Tweet_content"].apply(preprocess_text_data)
valid_df["clean_text"] = valid_df["Tweet_content"].apply(preprocess_text_data)

In [ ]:
train_df["length"] = train_df["clean_text"].apply(lambda x: len(x.split()))

In [ ]:
bins = [0, 10, 20, 30, 40, 50, 75, 100, 1000]

counts = pd.cut(train_df["length"], bins=bins).value_counts().sort_index()

print(counts)

length
(0, 10]        39491
(10, 20]       23284
(20, 30]        8329
(30, 40]        1102
(40, 50]          43
(50, 75]           3
(75, 100]         10
(100, 1000]        2
Name: count, dtype: int64


In [ ]:
print(train_df["length"].quantile([0.90, 0.95, 0.99]))

0.90    22.0
0.95    26.0
0.99    32.0
Name: length, dtype: float64


In [ ]:
label_map = {
    "Negative": 0,
    "Positive": 1,
    "Neutral": 2,
    "Irrelevant": 3
}

train_df["label"] = train_df["Sentiment"].map(label_map)
valid_df["label"] = valid_df["Sentiment"].map(label_map)

In [ ]:
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["clean_text"])

vocab_size = len(tokenizer.word_index) + 1
print(vocab_size)

26651


In [ ]:
X_train = tokenizer.texts_to_sequences(train_df["clean_text"])
X_test = tokenizer.texts_to_sequences(valid_df["clean_text"])

In [ ]:
maxlen = 32 #from my analysis this is supposed to be the best number

X_train = pad_sequences(X_train, maxlen=maxlen, padding="post", truncating="post")
X_test = pad_sequences(X_test, maxlen=maxlen, padding="post", truncating="post")

In [ ]:

y_train = to_categorical(train_df["label"], num_classes=4)
y_test = to_categorical(valid_df["label"], num_classes=4)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
)

#Task 1: Build the Baseline Model

In [ ]:
model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=maxlen
    ),

    SimpleRNN(32),

    Dense(4, activation="softmax")
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

training_time = time.time() - start

print("Training Time:", training_time)

Epoch 1/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 17s 7ms/step - accuracy: 0.6067 - loss: 0.9796 - val_accuracy: 0.7314 - val_loss: 0.7364
Epoch 2/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.8324 - loss: 0.4747 - val_accuracy: 0.7756 - val_loss: 0.6307
Epoch 3/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.8947 - loss: 0.2962 - val_accuracy: 0.7885 - val_loss: 0.5995
Epoch 4/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.9172 - loss: 0.2289 - val_accuracy: 0.7926 - val_loss: 0.6571
Epoch 5/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.9306 - loss: 0.1890 - val_accuracy: 0.7989 - val_loss: 0.6337
Training Time: 70.35102701187134


In [ ]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
print("Training Accuracy:", train_acc)

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print("Validation Accuracy:", val_acc)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", test_acc)

Training Accuracy: 0.946195662021637
Validation Accuracy: 0.7988513708114624
Test Accuracy: 0.9430000185966492


What is the role of the Embedding layer?
It learns meaningful vector representations of words during training, making the model to capture semantic relationships.

Why do we use SimpleRNN instead of a Dense layer?
Because rnn can capture context and word order and sequential dependencies.

What is the purpose of the Sigmoid activation function?
to overcome the linearity problem and gwt propabilities to make binary decisions.

#Task 2: Increase the Number of RNN Units

In [ ]:
model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=maxlen
    ),

    SimpleRNN(64),

    Dense(4, activation="softmax")
])

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

training_time = time.time() - start

print("Training Time:", training_time)

Epoch 1/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - accuracy: 0.6106 - loss: 0.9633 - val_accuracy: 0.7366 - val_loss: 0.7171
Epoch 2/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.8381 - loss: 0.4533 - val_accuracy: 0.7749 - val_loss: 0.6152
Epoch 3/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.8932 - loss: 0.3007 - val_accuracy: 0.7893 - val_loss: 0.6143
Epoch 4/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.9150 - loss: 0.2374 - val_accuracy: 0.7848 - val_loss: 0.6378
Epoch 5/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.9260 - loss: 0.2023 - val_accuracy: 0.7899 - val_loss: 0.6678
Training Time: 58.79069757461548


In [ ]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
print("Training Accuracy:", train_acc)

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print("Validation Accuracy:", val_acc)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", test_acc)

Training Accuracy: 0.9369890093803406
Validation Accuracy: 0.7898648381233215
Test Accuracy: 0.9300000071525574


Did the model performance improve?
slightly idecreased.

Did the training time increase?
No, it took less time.

Explain your observations.
it took less time with a slight decrement in accuracy.

#Task 3: Build a Stacked RNN

In [ ]:
model = Sequential([
    Embedding(vocab_size, 128, input_length=maxlen),

    SimpleRNN(32, return_sequences=True),

    SimpleRNN(32),

    Dense(4, activation="softmax")
])

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

training_time = time.time() - start
print("Training Time:", training_time)

Epoch 1/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 22s 9ms/step - accuracy: 0.6252 - loss: 0.9390 - val_accuracy: 0.7505 - val_loss: 0.6804
Epoch 2/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.8465 - loss: 0.4280 - val_accuracy: 0.7808 - val_loss: 0.6456
Epoch 3/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.9011 - loss: 0.2767 - val_accuracy: 0.8036 - val_loss: 0.5892
Epoch 4/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.9212 - loss: 0.2152 - val_accuracy: 0.8074 - val_loss: 0.5962
Epoch 5/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.9314 - loss: 0.1862 - val_accuracy: 0.7947 - val_loss: 0.6817
Training Time: 88.98282647132874


In [ ]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
print("Training Accuracy:", train_acc)

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print("Validation Accuracy:", val_acc)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", test_acc)

Training Accuracy: 0.940367579460144
Validation Accuracy: 0.7946621775627136
Test Accuracy: 0.9319999814033508


Why is return_sequences=True required?
so the output of layer goes to the one stacked above it

Did stacking RNN layers improve the results?
No, they decreased slightly.

Is adding more layers always beneficial? Explain.
No, not necessarily, it takes more time to achieve less accurate results.

#Task 4: Build a Bidirectional RNN

In [ ]:
model = Sequential([
    Embedding(vocab_size, 128, input_length=maxlen),

    Bidirectional(SimpleRNN(32)),

    Dense(4, activation="softmax")
])

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

training_time = time.time() - start
print("Training Time:", training_time)

Epoch 1/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 22s 10ms/step - accuracy: 0.6733 - loss: 0.8145 - val_accuracy: 0.8152 - val_loss: 0.5051
Epoch 2/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.8958 - loss: 0.2874 - val_accuracy: 0.8361 - val_loss: 0.4440
Epoch 3/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9332 - loss: 0.1787 - val_accuracy: 0.8459 - val_loss: 0.4477
Epoch 4/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9443 - loss: 0.1440 - val_accuracy: 0.8455 - val_loss: 0.4711
Epoch 5/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.9514 - loss: 0.1257 - val_accuracy: 0.8486 - val_loss: 0.4735
Training Time: 81.4733190536499


In [ ]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
print("Training Accuracy:", train_acc)

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print("Validation Accuracy:", val_acc)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", test_acc)

Training Accuracy: 0.9616696834564209
Validation Accuracy: 0.8486486673355103
Test Accuracy: 0.9610000252723694


Compare the results with the baseline model.
results improved especially validation accuracy.

Why might Bidirectional RNN achieve better performance?
use information from both the previous and the following words when making predictions, helping it better understand the context of the text. As a result, it often achieves higher accuracy than a standard SimpleRNN.

Is Bidirectional RNN suitable for real-time next-word prediction? Why or why not?
No, because it requires access to both past and future words in the sequence. During real-time text generation, future words are not yet available

#Task 5: Replace SimpleRNN with LSTM

In [ ]:
model = Sequential([
    Embedding(vocab_size, 128, input_length=maxlen),

    LSTM(32),

    Dense(4, activation="softmax")
])

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

training_time = time.time() - start
print("Training Time:", training_time)

Epoch 1/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 25s 10ms/step - accuracy: 0.5836 - loss: 1.0138 - val_accuracy: 0.7201 - val_loss: 0.7456
Epoch 2/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.8183 - loss: 0.5025 - val_accuracy: 0.8132 - val_loss: 0.5079
Epoch 3/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - accuracy: 0.8807 - loss: 0.3245 - val_accuracy: 0.8407 - val_loss: 0.4406
Epoch 4/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 21s 10ms/step - accuracy: 0.9072 - loss: 0.2477 - val_accuracy: 0.8533 - val_loss: 0.4383
Epoch 5/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - accuracy: 0.9224 - loss: 0.1989 - val_accuracy: 0.8611 - val_loss: 0.4382
Training Time: 100.08491635322571


In [ ]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
print("Training Accuracy:", train_acc)

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print("Validation Accuracy:", val_acc)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", test_acc)

Training Accuracy: 0.9392526745796204
Validation Accuracy: 0.861081063747406
Test Accuracy: 0.9390000104904175


Compare the performance with SimpleRNN.
the training accuracy and test accuracy had a slight negative change but the validation accuracy improved significantly
Which model required more training time?
LSTM took more time than simpleRNN
Why is LSTM generally better for long text sequences?
because it can keep important information over longer time steps using its memory cells and gating mechanisms (input, forget, and output gates). These gates help prevent the vanishing gradient problem, allowing the model to learn long-term dependencies more effectively than a SimpleRNN

#Task 6: Build a Bidirectional LSTM

In [ ]:
model = Sequential([
    Embedding(vocab_size, 128, input_length=maxlen),

    Bidirectional(LSTM(32)),

    Dense(4, activation="softmax")
])

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

training_time = time.time() - start
print("Training Time:", training_time)

Epoch 1/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 26s 13ms/step - accuracy: 0.6793 - loss: 0.8057 - val_accuracy: 0.7949 - val_loss: 0.5515
Epoch 2/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 41s 13ms/step - accuracy: 0.8574 - loss: 0.3850 - val_accuracy: 0.8384 - val_loss: 0.4352
Epoch 3/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 23s 13ms/step - accuracy: 0.9013 - loss: 0.2607 - val_accuracy: 0.8608 - val_loss: 0.3955
Epoch 4/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 23s 13ms/step - accuracy: 0.9220 - loss: 0.2024 - val_accuracy: 0.8636 - val_loss: 0.4167
Epoch 5/5
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - accuracy: 0.9336 - loss: 0.1680 - val_accuracy: 0.8652 - val_loss: 0.4313
Training Time: 137.17279267311096


In [ ]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
print("Training Accuracy:", train_acc)

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print("Validation Accuracy:", val_acc)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", test_acc)

Training Accuracy: 0.9446583986282349
Validation Accuracy: 0.8652027249336243
Test Accuracy: 0.9419999718666077


Which model achieved the highest accuracy?
Bidirectional RNN
Which model required the longest training time?
Bidirectional LSTM
Was the performance improvement worth the additional complexity? Explain.
Depends on the application, for exasmple for a medical application that doesn't require urgent actions the higher accuracy is worth the additional 11 seconds, but for a factory mass produciong goods i think time is priority.


#Task 7: Hyperparameter Experiment

In [ ]:
model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=maxlen
    ),

    SimpleRNN(32),

    Dense(4, activation="softmax")
])

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=64,
    verbose=1
)

training_time = time.time() - start

print("Training Time:", training_time)

Epoch 1/5
925/925 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - accuracy: 0.5822 - loss: 1.0138 - val_accuracy: 0.7024 - val_loss: 0.7907
Epoch 2/5
925/925 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8214 - loss: 0.5001 - val_accuracy: 0.7634 - val_loss: 0.6581
Epoch 3/5
925/925 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.8909 - loss: 0.3105 - val_accuracy: 0.7674 - val_loss: 0.6658
Epoch 4/5
925/925 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.9176 - loss: 0.2294 - val_accuracy: 0.7693 - val_loss: 0.7055
Epoch 5/5
925/925 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9319 - loss: 0.1881 - val_accuracy: 0.7667 - val_loss: 0.7539
Training Time: 33.65998196601868


In [ ]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
print("Training Accuracy:", train_acc)

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print("Validation Accuracy:", val_acc)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", test_acc)

Training Accuracy: 0.9448949098587036
Validation Accuracy: 0.7666891813278198
Test Accuracy: 0.9229999780654907


increasing batch size from 32 to 64 made a very slight improvement.

#Results Table


| Model | Test Accuracy | Training Time (s) | Observation |
|------|:-------------:|:-----------------:|------------|
| SimpleRNN | **94.30%** | 70.35 | Good baseline performance with moderate training time. |
| SimpleRNN (64 Units) | **93.00%** | 58.79 | Slightly lower accuracy; trained faster than the baseline. |
| Stacked RNN | **93.20%** | 88.98 | Increased training time without improving accuracy over the baseline. |
| Bidirectional RNN | **96.10%** | 81.47 | Highest test accuracy among the RNN models with a moderate increase in training time. |
| LSTM | **93.90%** | 100.08 | Longer training time than SimpleRNN, with comparable test accuracy but better validation accuracy. |
| Bidirectional LSTM | **94.20%** | 137.17 | Longest training time; achieved the highest validation accuracy but only a slight improvement in test accuracy over the baseline. |

#Bonus

##Experiment 1

In [ ]:
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Embedding(vocab_size, 128, input_length=maxlen),

    Bidirectional(LSTM(64, recurrent_dropout=0.2)),
    Dropout(0.5),

    Dense(32, activation="relu"),
    Dropout(0.3),

    Dense(4, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=2,
    restore_best_weights=True
)

start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

training_time = time.time() - start
print("Training Time:", training_time)

Epoch 1/15
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 470s 247ms/step - accuracy: 0.6340 - loss: 0.9166 - val_accuracy: 0.7675 - val_loss: 0.6191
Epoch 2/15
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 462s 250ms/step - accuracy: 0.8303 - loss: 0.4824 - val_accuracy: 0.8239 - val_loss: 0.4846
Epoch 3/15
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 402s 217ms/step - accuracy: 0.8761 - loss: 0.3459 - val_accuracy: 0.8434 - val_loss: 0.4442
Epoch 4/15
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 398s 215ms/step - accuracy: 0.8977 - loss: 0.2793 - val_accuracy: 0.8519 - val_loss: 0.4398
Epoch 5/15
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 439s 214ms/step - accuracy: 0.9100 - loss: 0.2434 - val_accuracy: 0.8640 - val_loss: 0.4316
Epoch 6/15
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 389s 210ms/step - accuracy: 0.9200 - loss: 0.2123 - val_accuracy: 0.8676 - val_loss: 0.4519
Epoch 7/15
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 390s 211ms/step - accuracy: 0.9294 - loss: 0.1886 - val_accuracy: 0.8726 - val_loss: 0.4725
Epoch 8/15
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 399s 216ms/step - ac

In [ ]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
print("Training Accuracy:", train_acc)

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print("Validation Accuracy:", val_acc)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", test_acc)

##Experiment 2

In [ ]:
from tensorflow.keras.layers import (
    Embedding,
    SimpleRNN,
    Dense,
    Bidirectional,
    Dropout
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    SimpleRNN,
    LSTM,
    Dense,
    Bidirectional,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Embedding(vocab_size, 128),
    Bidirectional(SimpleRNN(64)),
    Dropout(0.4),
    Dense(4, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=2,
    restore_best_weights=True
)

start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

training_time = time.time() - start
print("Training Time:", training_time)

Epoch 1/15
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 26s 11ms/step - accuracy: 0.6515 - loss: 0.8642 - val_accuracy: 0.7905 - val_loss: 0.5502
Epoch 2/15
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.8751 - loss: 0.3431 - val_accuracy: 0.8354 - val_loss: 0.4623
Epoch 3/15
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9190 - loss: 0.2207 - val_accuracy: 0.8359 - val_loss: 0.5129
Epoch 4/15
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.9351 - loss: 0.1771 - val_accuracy: 0.8459 - val_loss: 0.4705
Epoch 5/15
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9387 - loss: 0.1642 - val_accuracy: 0.8352 - val_loss: 0.5107
Epoch 6/15
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9450 - loss: 0.1478 - val_accuracy: 0.8396 - val_loss: 0.5201
Training Time: 99.60144901275635


In [ ]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
print("Training Accuracy:", train_acc)

val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print("Validation Accuracy:", val_acc)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", test_acc)

Training Accuracy: 0.9532231688499451
Validation Accuracy: 0.8458783626556396
Test Accuracy: 0.9520000219345093
